# 🌍 Robust Real-World Training Pipeline
### Domain-Gap Solution: Lab → Real World
**Framework:** PyTorch + Albumentations  
**Model:** EfficientNet-B2 (upgraded from B0)  
**Strategy:** Aggressive augmentation + Two-stage fine-tuning + TTA

---

## 🔍 Why Your Model Fails on Real Images

| Problem | Root Cause | Fix Applied |
|---------|-----------|-------------|
| Relies on white background | Never saw other backgrounds in training | `CoarseDropout` + `RandomShadow` |
| Fails in dim / harsh light | Only clean studio light in training | `CLAHE` + `RandomGamma` + `RandomShadow` |
| Fails on blurry phone photos | No blur/noise in training data | `GaussianBlur` + `GaussNoise` + `ISONoise` |
| Fails at odd angles | Always frontal shots in training | `Perspective` + `ShiftScaleRotate` |
| Overconfident wrong predictions | No label smoothing, frozen backbone | `label_smoothing=0.1` + full fine-tuning |
| Low accuracy despite training | B0 backbone permanently frozen | B2 + two-stage unfreeze strategy |

## ⚙️ Cell 1 — Install Dependencies

In [1]:
# Run once — installs albumentations (the star of this pipeline)
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "albumentations", "tqdm", "scikit-learn"
], check=True)
print("✅ Dependencies installed")

✅ Dependencies installed


## 📦 Cell 2 — Imports & Reproducibility

In [2]:
# ── Standard library ───────────────────────────────────────────────────────
import os, json, time, random, copy
from pathlib import Path
from collections import Counter
from typing import Optional

# ── Numeric / Visual ───────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# ── PyTorch ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.datasets import ImageFolder

# ── Albumentations ─────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Progress bar ───────────────────────────────────────────────────────────
from tqdm.auto import tqdm

# ── Metrics ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥  Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

c:\Users\benfa\miniconda3\envs\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥  Device: cuda
   GPU: NVIDIA GeForce RTX 4050 Laptop GPU
   VRAM: 6.4 GB


## 🗂️ Cell 3 — Config (Edit This)

> **Only cell you need to edit.** Set your paths and hyper-parameters here.

In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              ✏️  EDIT YOUR CONFIG HERE                      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Paths ──────────────────────────────────────────────────────────────────
# Point these to your already-split dataset folders.
# Each folder should contain one sub-folder per class with images inside.
#
#   dataset/
#   ├── train/
#   │   ├── class_a/  (*.jpg, *.png, ...)
#   │   └── class_b/
#   ├── val/
#   └── test/

TRAIN_DIR = "dataset/processed/train"   # ← change me
VAL_DIR   = "dataset/processed/val"     # ← change me
TEST_DIR  = "dataset/processed/test"    # ← change me
MODELS_DIR = "models/"
LOGS_DIR   = "logs/"
MODEL_NAME = "robust_model_v1"          # ← name for saved checkpoint files

# ── Model & image ──────────────────────────────────────────────────────────
IMG_SIZE    = 260    # EfficientNet-B2 native size. Use 224 if memory is tight.
BATCH_SIZE  = 32     # Reduce to 16 if you get CUDA OOM errors
NUM_WORKERS = min(4, os.cpu_count() or 1)

# ── Training stages ────────────────────────────────────────────────────────
# Stage 1: backbone frozen, only the new classifier head trains (fast warm-up)
STAGE1_EPOCHS = 5
STAGE1_LR     = 1e-3

# Stage 2: full model unfrozen, fine-tune end-to-end (main training)
STAGE2_EPOCHS = 25
STAGE2_LR     = 5e-4

# ── Regularisation ─────────────────────────────────────────────────────────
WEIGHT_DECAY      = 1e-4   # AdamW weight decay
LABEL_SMOOTHING   = 0.1    # Prevents overconfident wrong predictions
MIXUP_ALPHA       = 0.2    # MixUp strength (0.0 = off)
EARLY_STOP_PATIENCE = 7    # Stop if val acc doesn't improve for N epochs
DROPOUT           = 0.3    # Dropout in new classifier head

# ── Augmentation aggressiveness ────────────────────────────────────────────
# These multiply each group's probability. Set 0.5 for mild, 1.0 for full.
# Start at 1.0 and lower if you see train accuracy collapse below ~50%.
AUG_STRENGTH = 1.0

# ── Create output directories ──────────────────────────────────────────────
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(LOGS_DIR).mkdir(parents=True, exist_ok=True)

print("✅ Config loaded")
print(f"   Train: {TRAIN_DIR}")
print(f"   Val:   {VAL_DIR}")
print(f"   Test:  {TEST_DIR}")
print(f"   Model: EfficientNet-B2 @ {IMG_SIZE}px")
print(f"   Total epochs: {STAGE1_EPOCHS + STAGE2_EPOCHS}")

✅ Config loaded
   Train: dataset/processed/train
   Val:   dataset/processed/val
   Test:  dataset/processed/test
   Model: EfficientNet-B2 @ 260px
   Total epochs: 30


## 🔄 Cell 4 — Augmentation Pipelines

Each group targets a specific real-world failure mode:

- **Geometry** → viewing angle variation, partial crops  
- **Lighting** → shadows, overexposure, CLAHE  
- **Colour** → white balance drift, hue shifts  
- **Camera/Sensor** → blur, noise, JPEG compression  
- **Occlusion** → partially hidden objects

In [4]:
# ImageNet normalisation stats — correct for ALL EfficientNet variants
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def get_train_transform(img_size: int = 260, strength: float = 1.0) -> A.Compose:
    """
    Production augmentation pipeline that simulates real-world conditions.

    WHY Albumentations over torchvision.transforms?
      - Works on numpy arrays → faster CPU pipeline
      - 70+ domain-specific transforms (CLAHE, ISONoise, Perspective, etc.)
      - OneOf lets us randomly sample distortion types, not stack all of them
      - p-weighting is per-transform, not per-pipeline

    Args:
        img_size : model input resolution (260 for B2, 224 for B0/B1, 300 for B3)
        strength : multiply all probabilities by this factor (0.5 = mild, 1.0 = full)
    """
    s = strength  # shorthand

    return A.Compose([

        # ── 1. GEOMETRY ────────────────────────────────────────────────────
        # WHY: Your training set has objects always centred, frontal, and
        # at the same scale. Real photos have objects off-centre, rotated,
        # shot from below/above, at varying distances.
        A.RandomResizedCrop(
            height=img_size, width=img_size,
            scale=(0.65, 1.0),   # allow tight crops → teaches partial-object recognition
            ratio=(0.75, 1.33),
            p=1.0,               # always apply (replaces simple Resize)
        ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1 * s),   # lower prob — only if objects can appear upside-down
        A.Rotate(limit=30, border_mode=0, p=0.4 * s),
        A.Perspective(
            scale=(0.04, 0.10),  # subtle: simulates non-frontal shooting angle
            p=0.3 * s,
        ),
        A.ShiftScaleRotate(
            shift_limit=0.1, scale_limit=0.2, rotate_limit=0,
            border_mode=0,
            p=0.3 * s,
        ),

        # ── 2. LIGHTING ────────────────────────────────────────────────────
        # WHY: Studio images have controlled, even lighting.
        # Real photos have harsh shadows, over/underexposure, and
        # local contrast variations from windows, lamps, etc.
        A.OneOf([
            A.RandomGamma(gamma_limit=(60, 140)),           # over/under exposure
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8)), # local contrast enhancement
            A.RandomBrightnessContrast(
                brightness_limit=0.35,
                contrast_limit=0.35,
            ),
            A.RandomToneCurve(scale=0.15),                  # non-linear brightness
        ], p=0.65 * s),

        # Cast shadows — simulates objects/people partially blocking light
        A.RandomShadow(
            shadow_roi=(0, 0.3, 1, 1),
            num_shadows_lower=1,
            num_shadows_upper=2,
            shadow_dimension=5,
            p=0.3 * s,
        ),

        # ── 3. COLOUR / WHITE BALANCE ──────────────────────────────────────
        # WHY: Different cameras and lighting colour temperatures shift hue
        # and saturation. Indoor tungsten light makes images orange;
        # cloudy outdoor light makes them blue. The model must not depend
        # on exact colour values for classification.
        A.OneOf([
            A.HueSaturationValue(
                hue_shift_limit=20,
                sat_shift_limit=35,
                val_shift_limit=25,
            ),
            A.RGBShift(
                r_shift_limit=20,
                g_shift_limit=20,
                b_shift_limit=20,
            ),
            A.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1,
            ),
        ], p=0.5 * s),

        A.ToGray(p=0.04 * s),   # rare: simulate very low-light or greyscale captures

        # ── 4. CAMERA / SENSOR ARTEFACTS ──────────────────────────────────
        # WHY: Smartphone and budget cameras produce motion blur (shaky hands),
        # sensor noise (dark environments), and JPEG compression artefacts
        # (images shared over WhatsApp/social media). Your model has never
        # seen any of these.
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7)),   # camera shake / out of focus
            A.MotionBlur(blur_limit=9),          # fast movement or shaky hands
            A.MedianBlur(blur_limit=5),          # cheap sensor pixel smearing
        ], p=0.35 * s),

        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 60.0)),            # additive sensor noise
            A.ISONoise(
                color_shift=(0.01, 0.06),
                intensity=(0.1, 0.5),
            ),                                               # realistic ISO noise
            A.MultiplicativeNoise(
                multiplier=(0.85, 1.15),
                per_channel=True,
            ),                                               # pixel gain variation
        ], p=0.35 * s),

        # JPEG artefacts: images sent via phone apps are always re-compressed
        A.ImageCompression(
            quality_lower=55,
            quality_upper=95,
            p=0.3 * s,
        ),

        # ── 5. OCCLUSION / PARTIAL VISIBILITY ─────────────────────────────
        # WHY: In real scenes objects are partially hidden behind other objects,
        # cut by the image edge, or partially out of frame.
        # CoarseDropout forces the model to build distributed feature maps
        # instead of relying on one diagnostic region (e.g. a logo or a sticker).
        A.CoarseDropout(
            max_holes=8,
            max_height=img_size // 8,
            max_width=img_size // 8,
            min_holes=1,
            min_height=img_size // 32,
            min_width=img_size // 32,
            fill_value=0,   # black patches (most neutral fill)
            p=0.3 * s,
        ),

        # ── 6. NORMALISE (always last) ─────────────────────────────────────
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_val_transform(img_size: int = 260) -> A.Compose:
    """
    Deterministic val/test transform — NO augmentation.

    IMPORTANT: Never augment validation. If you do, loss curves become noisy,
    you can't detect overfitting, and val accuracy underestimates real accuracy.
    """
    return A.Compose([
        A.Resize(img_size + 32, img_size + 32),
        A.CenterCrop(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print("✅ Augmentation pipelines defined")
print(f"   Train: 5 augmentation groups (geometry, lighting, colour, camera, occlusion)")
print(f"   Val/Test: resize + centre crop + normalise only")

✅ Augmentation pipelines defined
   Train: 5 augmentation groups (geometry, lighting, colour, camera, occlusion)
   Val/Test: resize + centre crop + normalise only


## 🔬 Cell 5 — Preview Augmentations

Before training, visually check that augmentations look realistic — not so aggressive that the images become unrecognisable.

In [5]:
def preview_augmentations(
    image_path: str,
    n_versions: int = 8,
    img_size: int = 260,
    strength: float = 1.0,
):
    """
    Show one original image alongside n_versions augmented copies.
    Run this BEFORE training to sanity-check augmentation aggressiveness.

    If the augmented versions look completely unrecognisable → lower AUG_STRENGTH.
    If they all look almost identical to the original → raise AUG_STRENGTH.
    """
    img_np  = np.array(PILImage.open(image_path).convert("RGB"))
    tfm     = get_train_transform(img_size, strength)

    # Undo normalisation for display
    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)

    def to_display(t):
        arr = t.permute(1, 2, 0).numpy()
        arr = (arr * std + mean).clip(0, 1)
        return arr

    cols = 4
    rows = (n_versions + cols) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    # Original
    original_resized = A.Compose([
        A.Resize(img_size, img_size),
    ])(image=img_np)["image"]
    axes[0].imshow(original_resized)
    axes[0].set_title("Original", fontsize=10, color="green", fontweight="bold")
    axes[0].axis("off")

    # Augmented versions
    for i in range(1, n_versions + 1):
        augmented = tfm(image=img_np)["image"]
        axes[i].imshow(to_display(augmented))
        axes[i].set_title(f"Aug #{i}", fontsize=9)
        axes[i].axis("off")

    for ax in axes[n_versions + 1:]:
        ax.axis("off")

    plt.suptitle(
        f"Augmentation preview (strength={strength})\n"
        "All look like plausible real-world images? Good. "
        "Too extreme? Lower AUG_STRENGTH.",
        fontsize=10
    )
    plt.tight_layout()
    plt.savefig(f"{LOGS_DIR}/augmentation_preview.png", dpi=120)
    plt.show()


# ── Usage: replace with any image path from your dataset ──────────────────
# preview_augmentations("dataset/processed/train/your_class/image.jpg")

# Auto-find a sample image if dataset exists
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
sample = next(
    (p for p in Path(TRAIN_DIR).rglob("*") if p.is_file() and p.suffix.lower() in valid_exts),
    None,
)

if sample is not None:
    preview_augmentations(str(sample), n_versions=8, img_size=IMG_SIZE, strength=AUG_STRENGTH)
else:
    print("⚠️  No images found at TRAIN_DIR — set the correct path in Cell 3.")
    print("    Then run: preview_augmentations('your/image/path.jpg')")

⚠️  No images found at TRAIN_DIR — set the correct path in Cell 3.
    Then run: preview_augmentations('your/image/path.jpg')


## 🗄️ Cell 6 — Dataset & DataLoaders

In [6]:
class AlbumentationsDataset(torch.utils.data.Dataset):
    """
    Wraps torchvision ImageFolder so it accepts an Albumentations pipeline.

    ImageFolder returns PIL images. Albumentations expects numpy uint8 [H,W,C].
    This class bridges the two.
    """
    def __init__(self, root: str, transform: A.Compose):
        self._base         = ImageFolder(root)
        self.transform     = transform
        self.classes       = self._base.classes
        self.class_to_idx  = self._base.class_to_idx
        self.samples       = self._base.samples

    def __len__(self) -> int:
        return len(self._base)

    def __getitem__(self, idx: int):
        img_pil, label = self._base[idx]
        img_np         = np.array(img_pil)                # PIL → numpy uint8
        augmented      = self.transform(image=img_np)
        return augmented["image"], label                  # Tensor[C,H,W], int


# ── Build datasets ─────────────────────────────────────────────────────────
train_dataset = AlbumentationsDataset(TRAIN_DIR, get_train_transform(IMG_SIZE, AUG_STRENGTH))
val_dataset   = AlbumentationsDataset(VAL_DIR,   get_val_transform(IMG_SIZE))
test_dataset  = AlbumentationsDataset(TEST_DIR,  get_val_transform(IMG_SIZE))

NUM_CLASSES = len(train_dataset.classes)
CLASS_NAMES = train_dataset.classes

print(f"✅ {NUM_CLASSES} classes: {CLASS_NAMES}")
print(f"   Train: {len(train_dataset)} images")
print(f"   Val:   {len(val_dataset)} images")
print(f"   Test:  {len(test_dataset)} images")

# ── Class distribution chart ───────────────────────────────────────────────
targets  = [s[1] for s in train_dataset.samples]
counts   = Counter(targets)
fig, ax  = plt.subplots(figsize=(max(8, NUM_CLASSES * 0.8), 4))
bars = ax.bar(
    CLASS_NAMES,
    [counts[i] for i in range(NUM_CLASSES)],
    color="steelblue", edgecolor="white"
)
ax.set_title("Training class distribution (before weighted sampling)")
ax.set_ylabel("Images")
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
            str(int(b.get_height())), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(f"{LOGS_DIR}/class_distribution.png", dpi=120)
plt.show()

# ── Weighted sampler — fixes class imbalance ───────────────────────────────
# Without this, the model sees majority classes far more often and ignores
# minority classes. WeightedRandomSampler rebalances without losing any data.
sample_weights = [1.0 / counts[t] for t in targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

# ── DataLoaders ────────────────────────────────────────────────────────────
pin = (DEVICE.type == "cuda")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler,             # replaces shuffle=True; balanced per batch
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    drop_last=True,              # prevents BatchNorm errors on last tiny batch
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin,
)

print(f"\n📦 Batches per epoch: {len(train_loader)}")

ValueError: 1 validation error for InitSchema
size
  Field required [type=missing, input_value={'scale': (0.65, 1.0), 'r...: None, 'strict': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

## 🧠 Cell 7 — Model: EfficientNet-B2

**Why upgrade from B0 → B2?**
- B0 (~5M params, 224px) was designed for speed. B2 (~9M params, 260px) captures finer texture and colour details — critical for date fruit classification.
- Permanently freezing the backbone (your v1 approach) means the model can only learn shallow remapping of ImageNet features, never adapting to date-specific features.
- The **two-stage strategy** is the key upgrade: warm up the head (Stage 1), then unlock everything and fine-tune end-to-end (Stage 2).

In [ ]:
def build_model(
    num_classes:     int,
    freeze_backbone: bool  = True,
    dropout:         float = 0.3,
) -> nn.Module:
    """
    EfficientNet-B2 with a regularised two-layer classifier head.

    Head architecture:  Dropout → Linear(1408→512) → BN → SiLU → Dropout → Linear(512→N)

    The extra Linear + BN layer vs the original single-layer head:
      • More capacity for fine-grained visual differences between classes
      • BatchNorm stabilises training when unfreezing the backbone
      • SiLU (same activation as EfficientNet internals) for consistency

    Args:
        num_classes     : your number of output classes
        freeze_backbone : True = Stage 1 (only head trains)
        dropout         : dropout rate in the new head
    """
    # Use the new weights API (avoids deprecation warning from pretrained=True)
    weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
    model   = models.efficientnet_b2(weights=weights)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace the original single-layer classifier
    in_features = model.classifier[1].in_features  # 1408 for B2
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=dropout * 0.67),
        nn.Linear(512, num_classes),
    )
    return model


def unfreeze_backbone(model: nn.Module) -> None:
    """Unfreeze all parameters for Stage 2 end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"🔓 Backbone unfrozen — {n:,} trainable parameters")


# ── Build model (Stage 1: backbone frozen) ────────────────────────────────
model = build_model(NUM_CLASSES, freeze_backbone=True, dropout=DROPOUT).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\n🔢 Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"   (backbone frozen — Stage 1 will train only the classifier head)")

## 🎲 Cell 8 — MixUp Helper

MixUp linearly blends pairs of training images and their labels. It acts as a strong regulariser and is especially effective for fine-grained classification. It teaches the model to produce smooth, calibrated predictions rather than hard boundaries.

In [ ]:
def mixup_data(
    x: torch.Tensor,
    y: torch.Tensor,
    alpha: float = 0.2,
):
    """
    Apply MixUp to a training batch.

    Returns:
        mixed_images, labels_a, labels_b, lambda
    Set alpha=0.0 to disable entirely.
    """
    if alpha <= 0.0:
        return x, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_loss(
    criterion: nn.Module,
    preds:     torch.Tensor,
    y_a:       torch.Tensor,
    y_b:       torch.Tensor,
    lam:       float,
) -> torch.Tensor:
    """Compute MixUp loss = λ·CE(y_a) + (1-λ)·CE(y_b)."""
    return lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)


print(f"✅ MixUp helper defined  (alpha={MIXUP_ALPHA})")

## 🔁 Cell 9 — Training & Validation Loop Functions

In [ ]:
def train_one_epoch(
    model:        nn.Module,
    loader:       DataLoader,
    criterion:    nn.Module,
    optimizer:    torch.optim.Optimizer,
    scheduler:    torch.optim.lr_scheduler._LRScheduler,
    device:       torch.device,
    mixup_alpha:  float = 0.2,
    scaler=None,
) -> tuple:
    """
    One full training epoch.

    Includes:
      • MixUp data augmentation (optional)
      • AMP (Automatic Mixed Precision) for ~2× speed on CUDA
      • Gradient clipping (max_norm=1.0) prevents exploding gradients
      • OneCycleLR steps per batch (not per epoch)

    Returns: (avg_loss, accuracy_percent)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="  train", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # MixUp: blend pairs of images
        images, y_a, y_b, lam = mixup_data(images, labels, alpha=mixup_alpha)

        optimizer.zero_grad()

        if scaler is not None:                           # AMP path (CUDA)
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:                                            # CPU / non-AMP path
            logits = model(images)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()     # OneCycleLR must step every batch

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> tuple:
    """
    Full evaluation pass — no augmentation, no MixUp.
    Returns: (avg_loss, accuracy_percent)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="  eval ", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), 100.0 * correct / total


print("✅ Training / evaluation functions defined")

## 🚀 Cell 10 — Stage 1: Train Classifier Head Only

The backbone is frozen. Only the new 2-layer classifier head trains.  
**Purpose:** Warm up the head weights to a reasonable starting point before unleashing the full model.

In [ ]:
# ── Loss function ─────────────────────────────────────────────────────────
# CrossEntropy + label_smoothing=0.1:
#   Converts hard targets [0,1,0,...] to soft [0.011, 0.89, 0.011,...]
#   Prevents the model from becoming overconfident on training examples,
#   which is the main cause of poor calibration on real-world images.
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(DEVICE)

# AMP scaler — ~2× faster training on CUDA, no accuracy loss
scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

# ── Stage 1 optimiser ─────────────────────────────────────────────────────
# AdamW over Adam: better weight decay (decoupled from gradient scaling)
optimizer_s1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=STAGE1_LR,
    weight_decay=WEIGHT_DECAY,
)
# OneCycleLR: ramps LR up then down in a cosine curve over the stage
# Much better than fixed LR or StepLR for fine-tuning
scheduler_s1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_s1,
    max_lr=STAGE1_LR,
    steps_per_epoch=len(train_loader),
    epochs=STAGE1_EPOCHS,
)

# ── Training history ──────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

BEST_VAL_ACC  = 0.0
BEST_CKPT     = Path(MODELS_DIR) / f"{MODEL_NAME}_best.pth"
patience_left = EARLY_STOP_PATIENCE
global_epoch  = 0

print(f"{'='*65}")
print(f"  STAGE 1 — {STAGE1_EPOCHS} epochs | lr={STAGE1_LR} | backbone frozen")
print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")
print(f"{'='*65}\n")

for _ in range(STAGE1_EPOCHS):
    global_epoch += 1
    t0 = time.time()

    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_s1, scheduler_s1,
        DEVICE, mixup_alpha=0.0,          # no MixUp in stage 1 (head too raw)
        scaler=scaler,
    )
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["lr"].append(optimizer_s1.param_groups[0]["lr"])

    flag = ""
    if vl_acc > BEST_VAL_ACC:
        BEST_VAL_ACC = vl_acc
        # Save full checkpoint: weights + metadata
        torch.save({
            "epoch":       global_epoch,
            "model_state": model.state_dict(),
            "val_acc":     vl_acc,
            "class_names": CLASS_NAMES,
            "img_size":    IMG_SIZE,
        }, BEST_CKPT)
        flag = "  ★ best saved"
        patience_left = EARLY_STOP_PATIENCE
    else:
        patience_left -= 1

    print(
        f"  Ep {global_epoch:2d}/{STAGE1_EPOCHS}  "
        f"TrLoss={tr_loss:.4f}  TrAcc={tr_acc:5.1f}%  "
        f"VlLoss={vl_loss:.4f}  VlAcc={vl_acc:5.1f}%  "
        f"[{time.time()-t0:.0f}s]{flag}"
    )

    if patience_left == 0:
        print("  ⏹  Early stopping triggered")
        break

print(f"\n✅ Stage 1 done. Best val acc: {BEST_VAL_ACC:.2f}%")

## 🔓 Cell 11 — Stage 2: Unfreeze & Fine-Tune End-to-End

Now we unfreeze the entire backbone and train everything with a lower LR.  
This is where the model learns date-specific visual features, not just ImageNet remapping.

In [ ]:
# ── Unfreeze backbone ─────────────────────────────────────────────────────
unfreeze_backbone(model)
patience_left = EARLY_STOP_PATIENCE    # reset patience counter

# ── Stage 2 optimiser ─────────────────────────────────────────────────────
optimizer_s2 = torch.optim.AdamW(
    model.parameters(),
    lr=STAGE2_LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler_s2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_s2,
    max_lr=STAGE2_LR,
    steps_per_epoch=len(train_loader),
    epochs=STAGE2_EPOCHS,
)

STAGE1_END = global_epoch   # save for curve plot separator

print(f"{'='*65}")
print(f"  STAGE 2 — {STAGE2_EPOCHS} epochs | max_lr={STAGE2_LR} | full model")
print(f"  MixUp alpha={MIXUP_ALPHA}")
print(f"{'='*65}\n")

for _ in range(STAGE2_EPOCHS):
    global_epoch += 1
    t0 = time.time()

    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_s2, scheduler_s2,
        DEVICE, mixup_alpha=MIXUP_ALPHA,   # MixUp active in stage 2
        scaler=scaler,
    )
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["lr"].append(optimizer_s2.param_groups[0]["lr"])

    flag = ""
    if vl_acc > BEST_VAL_ACC:
        BEST_VAL_ACC = vl_acc
        torch.save({
            "epoch":       global_epoch,
            "model_state": model.state_dict(),
            "val_acc":     vl_acc,
            "class_names": CLASS_NAMES,
            "img_size":    IMG_SIZE,
        }, BEST_CKPT)
        flag = "  ★ best saved"
        patience_left = EARLY_STOP_PATIENCE
    else:
        patience_left -= 1

    total_ep = STAGE1_EPOCHS + STAGE2_EPOCHS
    print(
        f"  Ep {global_epoch:2d}/{total_ep}  "
        f"TrLoss={tr_loss:.4f}  TrAcc={tr_acc:5.1f}%  "
        f"VlLoss={vl_loss:.4f}  VlAcc={vl_acc:5.1f}%  "
        f"[{time.time()-t0:.0f}s]{flag}"
    )

    if patience_left == 0:
        print("  ⏹  Early stopping triggered")
        break

print(f"\n✅ Training complete!")
print(f"   Best val accuracy: {BEST_VAL_ACC:.2f}%")
print(f"   Best checkpoint:   {BEST_CKPT}")

## 📈 Cell 12 — Training Curves

In [ ]:
n_ep  = len(history["train_loss"])
xs    = range(1, n_ep + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training history", fontsize=13)

# Loss
axes[0].plot(xs, history["train_loss"], label="Train",      marker="o", ms=3)
axes[0].plot(xs, history["val_loss"],   label="Validation", marker="o", ms=3)
axes[0].axvline(STAGE1_END, color="gray", ls="--", alpha=0.6, label="S1→S2")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(xs, history["train_acc"], label="Train",      marker="o", ms=3)
axes[1].plot(xs, history["val_acc"],   label="Validation", marker="o", ms=3)
axes[1].axvline(STAGE1_END, color="gray", ls="--", alpha=0.6, label="S1→S2")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning rate
axes[2].semilogy(xs, history["lr"], color="darkorange")
axes[2].axvline(STAGE1_END, color="gray", ls="--", alpha=0.6, label="S1→S2")
axes[2].set_title("Learning Rate (log scale)")
axes[2].set_xlabel("Epoch")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{LOGS_DIR}/training_curves.png", dpi=120)
plt.show()
print(f"Saved → {LOGS_DIR}/training_curves.png")

## 🧪 Cell 13 — Test Set Evaluation

> **Important:** We load the **best checkpoint** (not the last epoch) before evaluating on test.

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"📂 Loaded best checkpoint — epoch {ckpt['epoch']}, val acc {ckpt['val_acc']:.2f}%")

# ── Standard test loop ────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE, non_blocking=True)
        preds  = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = 100.0 * (all_preds == all_labels).mean()
print(f"\n🎯 Test Accuracy (standard): {test_acc:.2f}%")
print("\n" + classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    digits=3,
))

## 🔮 Cell 14 — Test-Time Augmentation (TTA)

TTA runs **5 forward passes** per image with mild random crops/flips, then **averages the softmax probabilities**. This is a free accuracy boost (no extra training) — typically +2–5% on real-world images.

In [ ]:
def tta_predict(
    model:      nn.Module,
    loader:     DataLoader,
    device:     torch.device,
    img_size:   int = 260,
    n_augments: int = 5,
) -> tuple:
    """
    Test-time augmentation: ensemble multiple views of each test image.

    WHY: A single crop of a real-world image might show the object from a
    slightly unfavourable angle or cut off a key feature. By averaging
    predictions over 5 mildly different crops, we get a more stable
    prediction that is less sensitive to positioning.

    Args:
        n_augments: number of views per image (5 is the sweet spot)

    Returns: (all_preds, all_labels)
    """
    # Mild TTA transform — subtle enough to not corrupt the prediction
    tta_tfm = A.Compose([
        A.RandomResizedCrop(img_size, img_size, scale=(0.85, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])
    val_tfm = get_val_transform(img_size)   # deterministic centre-crop view

    mean_np = np.array(IMAGENET_MEAN)
    std_np  = np.array(IMAGENET_STD)

    model.eval()
    all_preds, all_labels = [], []

    for images_batch, labels in tqdm(loader, desc="TTA"):
        bsz    = images_batch.size(0)
        # n_classes inferred from last Linear layer
        n_cls  = model.classifier[-1].out_features
        accum  = torch.zeros(bsz, n_cls).to(device)

        for aug_idx in range(n_augments):
            aug_batch = []
            for img_t in images_batch:
                # Denormalise tensor back to uint8 numpy
                img_np = img_t.permute(1, 2, 0).numpy()
                img_np = (img_np * std_np + mean_np)
                img_np = (img_np * 255).clip(0, 255).astype(np.uint8)

                # View 0 = deterministic centre crop; 1+ = random crops
                tfm = val_tfm if aug_idx == 0 else tta_tfm
                aug_batch.append(tfm(image=img_np)["image"])

            aug_tensor = torch.stack(aug_batch).to(device)
            with torch.no_grad():
                accum += F.softmax(model(aug_tensor), dim=1)

        all_preds.extend(accum.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


# ── Run TTA ───────────────────────────────────────────────────────────────
tta_preds, tta_labels = tta_predict(
    model, test_loader, DEVICE,
    img_size=IMG_SIZE, n_augments=5,
)

tta_acc = 100.0 * (tta_preds == tta_labels).mean()
print(f"\n🎯 Test Accuracy with TTA (5 views): {tta_acc:.2f}%")
print(f"   Improvement over standard:         +{tta_acc - test_acc:.2f}%")
print("\n" + classification_report(
    tta_labels, tta_preds,
    target_names=CLASS_NAMES,
    digits=3,
))

## 🗺️ Cell 15 — Confusion Matrix

In [ ]:
cm      = confusion_matrix(tta_labels, tta_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(max(14, NUM_CLASSES * 1.2),
                                        max(6,  NUM_CLASSES * 0.9)))

ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], xticks_rotation=45, colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion matrix — raw counts")

ConfusionMatrixDisplay(np.round(cm_norm, 2), display_labels=CLASS_NAMES).plot(
    ax=axes[1], xticks_rotation=45, colorbar=False, cmap="Blues"
)
axes[1].set_title("Confusion matrix — normalised (row = recall per class)")

plt.tight_layout()
plt.savefig(f"{LOGS_DIR}/confusion_matrix.png", dpi=120)
plt.show()

# ── Identify worst-performing pairs ───────────────────────────────────────
print("\n🔍 Most confused pairs (off-diagonal, normalised):")
n = cm_norm.shape[0]
pairs = []
for i in range(n):
    for j in range(n):
        if i != j and cm_norm[i, j] > 0:
            pairs.append((cm_norm[i, j], CLASS_NAMES[i], CLASS_NAMES[j]))
for rate, true_cls, pred_cls in sorted(pairs, reverse=True)[:5]:
    print(f"   True={true_cls:20s} → Predicted={pred_cls:20s}  ({rate*100:.1f}% of true class)")

## 💾 Cell 16 — Save Final Model + Metadata

In [ ]:
# ── Final checkpoint (last epoch weights) ────────────────────────────────
final_ckpt = Path(MODELS_DIR) / f"{MODEL_NAME}_final.pth"
torch.save({
    "epoch":       global_epoch,
    "model_state": model.state_dict(),
    "val_acc":     history["val_acc"][-1],
    "class_names": CLASS_NAMES,
    "img_size":    IMG_SIZE,
}, final_ckpt)

# ── Metadata JSON ─────────────────────────────────────────────────────────
metadata = {
    "model":          "efficientnet_b2",
    "num_classes":    NUM_CLASSES,
    "class_names":    CLASS_NAMES,
    "best_val_acc":   round(BEST_VAL_ACC, 4),
    "test_acc":       round(test_acc, 4),
    "test_acc_tta":   round(tta_acc, 4),
    "epochs_ran":     global_epoch,
    "stage1_epochs":  STAGE1_END,
    "img_size":       IMG_SIZE,
    "img_mean":       list(IMAGENET_MEAN),
    "img_std":        list(IMAGENET_STD),
    "mixup_alpha":    MIXUP_ALPHA,
    "label_smoothing": LABEL_SMOOTHING,
    "aug_strength":   AUG_STRENGTH,
}
meta_path = Path(MODELS_DIR) / f"{MODEL_NAME}_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))

print("✅ Saved:")
print(f"   Best checkpoint  → {BEST_CKPT}")
print(f"   Final checkpoint → {final_ckpt}")
print(f"   Metadata         → {meta_path}")
print(f"\n🏆 Summary:")
print(f"   Best Val Acc  : {BEST_VAL_ACC:.2f}%")
print(f"   Test Acc      : {test_acc:.2f}%")
print(f"   Test Acc (TTA): {tta_acc:.2f}%")

## 🔍 Cell 17 — Single Image Inference

Use this to test the deployed model on any real-world image.

In [ ]:
def predict_image(
    image_path: str,
    ckpt_path:  str = str(BEST_CKPT),
    use_tta:    bool = True,
) -> dict:
    """
    Classify a single real-world image.

    Args:
        image_path : any image file (.jpg, .png, etc.)
        ckpt_path  : path to .pth checkpoint (defaults to best checkpoint)
        use_tta    : if True, averages 5 views for better accuracy

    Returns:
        dict: {'class': str, 'confidence': float, 'top5_probs': dict}
    """
    ckpt        = torch.load(ckpt_path, map_location=DEVICE)
    cls_names   = ckpt.get("class_names", CLASS_NAMES)
    n_cls       = len(cls_names)
    sz          = ckpt.get("img_size", IMG_SIZE)

    # Rebuild model and load weights
    inf_model = build_model(n_cls, freeze_backbone=False).to(DEVICE)
    inf_model.load_state_dict(ckpt["model_state"])
    inf_model.eval()

    img_np  = np.array(PILImage.open(image_path).convert("RGB"))
    val_tfm = get_val_transform(sz)
    tta_tfm = A.Compose([
        A.RandomResizedCrop(sz, sz, scale=(0.85, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

    if use_tta:
        views = [val_tfm(image=img_np)["image"]]         # view 0: deterministic
        for _ in range(4):                               # views 1-4: random
            views.append(tta_tfm(image=img_np)["image"])
        batch = torch.stack(views).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(inf_model(batch), dim=1).mean(0)
    else:
        tensor = val_tfm(image=img_np)["image"].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(inf_model(tensor), dim=1).squeeze()

    top5_v, top5_i = probs.topk(min(5, n_cls))
    top5 = {cls_names[i]: round(v.item(), 4) for i, v in zip(top5_i, top5_v)}

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(PILImage.open(image_path).convert("RGB"))
    axes[0].set_title(f"Input image", fontsize=10)
    axes[0].axis("off")
    axes[1].barh(
        list(top5.keys())[::-1],
        [v * 100 for v in list(top5.values())[::-1]],
        color="steelblue"
    )
    axes[1].set_xlabel("Confidence (%)")
    axes[1].set_title(f"Prediction: {cls_names[top5_i[0]]} ({top5_v[0]*100:.1f}%)")
    axes[1].set_xlim(0, 100)
    plt.tight_layout()
    plt.show()

    return {
        "class":       cls_names[top5_i[0].item()],
        "confidence":  round(top5_v[0].item() * 100, 2),
        "top5_probs":  top5,
    }


# ── Sanity check on a random test image ──────────────────────────────────
try:
    sample_class = next(Path(TEST_DIR).iterdir())
    sample_img   = next(sample_class.iterdir())
    result = predict_image(str(sample_img), use_tta=True)
    print(f"True class    : {sample_class.name}")
    print(f"Predicted     : {result['class']} ({result['confidence']:.1f}%)")
    print(f"Top-5 probs   : {result['top5_probs']}")
except Exception as e:
    print(f"⚠️  Set TEST_DIR correctly to test inference. Error: {e}")
    print("    Usage: result = predict_image('path/to/your/image.jpg')")